# NB2 · Fine-tuning LoRA / QLoRA → đánh giá → Hybrid-RAG → xuất mô hình → demo MVP
**Chatbot CSKH Kimochi Sushi.** Chạy trên **Kaggle GPU T4** (Settings: Accelerator *GPU T4 x2*, Internet *On*; Add Data: dataset
chứa repo này, đã chạy NB1; Secrets: `HF_TOKEN`, tùy chọn `GEMINI_API_KEY`). Dùng *Save & Run All (Commit)* để chạy nền.

| § | Nội dung | Thời gian ước tính (T4) |
|---|---|---|
| 0 | Cấu hình thí nghiệm | — |
| 1 | Nhận dữ liệu & kiểm tra manifest | 1 phút |
| 2 | Thiết lập: siêu tham số (Optuna tùy chọn), mô hình cơ sở | 30–60 phút |
| 3 | Fine-tune lần lượt **LoRA 16-bit**, **QLoRA 4-bit** (+ QLoRA 8-bit tùy chọn) × {Qwen, LLaMA} | 3–6 giờ |
| 4 | Đánh giá các mô hình trên tập test | 40–80 phút |
| 5 | Trực quan & phân tích kết quả | 2 phút |
| 6 | Demo giao diện nhanh (mô hình tốt nhất) | tùy chọn |
| 7 | Xây dựng Hybrid-RAG (BM25 + dense + RRF + reranker + lọc) & ablation | 10–15 phút |
| 8 | Đánh giá hệ thống đầy đủ: closed-book ↔ RAG ↔ RAG + Agent; so sánh, trực quan | 30–60 phút |
| 9 | Xuất mô hình hoàn chỉnh (gộp adapter) + gói MVP | 10 phút |
| 10 | Demo MVP | — |

Thời gian là **ước lượng**. Phiên Kaggle tối đa 12 giờ → mọi bước **tự bỏ qua nếu đã có kết quả** (`RESUME_GLOB`): chạy lại
notebook với output phiên trước làm input là chạy tiếp. Kết quả ghi vào `results/` với cùng tên file mà báo cáo đọc.

In [ ]:
%%capture
!pip install -q "transformers==4.56.2" "trl==0.21.0" "peft==0.17.0" "bitsandbytes==0.48.2" "accelerate>=1.4" "datasets>=3.0" \
                rank-bm25 "sentence-transformers>=3.0" optuna gradio==5.49.1 google-genai

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # chỉ dùng 1 GPU (xem kaggle_env.py) — phải đặt trước khi import torch
os.environ.update({'USE_TF': '0', 'USE_JAX': '0', 'TF_CPP_MIN_LOG_LEVEL': '3',
                   'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'})   # giảm phân mảnh bộ nhớ GPU   # không nạp TensorFlow (chỉ dùng PyTorch)
import sys, glob, json, time, shutil, pathlib
cands = glob.glob('/kaggle/input/**/code/src', recursive=True)
if cands:                       # Kaggle: chép repo (dataset chỉ đọc) sang /kaggle/working/repo
    sys.path.insert(0, cands[0])
    from kimochi.kaggle_env import setup, gpu_info
    ROOT = setup()
else:                           # local / Colab: tìm thư mục gốc repo
    ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'code/src/kimochi').exists())
    sys.path.insert(0, str(ROOT / 'code/src'))
    from kimochi.kaggle_env import gpu_info
os.chdir(ROOT)
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image
from kimochi import llm, prompts, metrics, retriever, agent, agent_eval, viz, manifest, tools
RES = ROOT / 'results'; GEN = RES / 'gen'; GEN.mkdir(parents=True, exist_ok=True)
WORK = pathlib.Path('/kaggle/working') if pathlib.Path('/kaggle/working').exists() else ROOT / 'runs'
RUNS = WORK / 'runs'; RUNS.mkdir(parents=True, exist_ok=True)
print('GPU:', gpu_info())

## 0. Cấu hình thí nghiệm
Chỉ đổi mô hình nền và kiểu lượng tử; mọi thứ khác giống nhau (CLAUDE.md §3.5).

In [ ]:
MODELS  = ['qwen', 'llama']                                   # Qwen2.5-3B-Instruct, LLaMA-3.2-3B-Instruct
METHODS = {'LoRA 16-bit': 'fp16', 'QLoRA 4-bit': 'nf4'}       # thêm 'QLoRA 8-bit': 'int8' nếu đủ thời gian GPU
SEED, EPOCHS, MAX_LEN = 42, 2, 1024
DEFAULT_HP = {'r': 16, 'lr': 2e-4}
RUN_OPTUNA, N_TRIALS = False, 6                               # True: dò r, lr trên Qwen QLoRA 4-bit (~1,5–2 giờ)
RUN_BASELINE = True                                           # B0 zero-shot của mô hình gốc để so sánh
N_EVAL = None                                                 # None = toàn bộ tập test; 150 để chạy thử nhanh
RUN_QUICK_DEMO = False                                        # §6: mở demo nhanh (không chặn notebook)
RUN_MVP_DEMO = False                                          # §10 — CHỈ bật khi chạy tương tác (ô demo chạy mãi, tốn GPU)
HF_USER = None                                                # vd 'thanhtrung665' → đẩy adapter + mô hình gộp lên HF Hub (private)
RESUME_GLOB = '/kaggle/input/**/runs/*/result.json'           # kết quả huấn luyện phiên trước (nếu có)
for p in glob.glob(RESUME_GLOB, recursive=True):              # nạp lại ô đã huấn luyện
    src = pathlib.Path(p).parent
    if not (RUNS / src.name).exists():
        shutil.copytree(src, RUNS / src.name, ignore=shutil.ignore_patterns('checkpoint-*'))
for p in glob.glob('/kaggle/input/**/repo/results', recursive=True):   # kết quả + câu trả lời đã sinh của phiên trước
    shutil.copytree(p, RES, dirs_exist_ok=True)
print('Ô đã có:', sorted(x.name for x in RUNS.iterdir()), '| file kết quả:', len(list(RES.rglob('*.*'))))

## 1. Nhận dữ liệu
Kiểm tra md5 so với `data/MANIFEST.json` do NB1 ghi — bảo đảm huấn luyện đúng phiên bản dữ liệu.

In [ ]:
bad = manifest.verify(ROOT)
assert not bad, f'Dữ liệu lệch manifest: {bad} → chạy lại NB1 rồi tạo version dataset mới'
print('✓ Dữ liệu khớp manifest')
train_ds, val_ds = llm.sft_dataset(ROOT, 'train'), llm.sft_dataset(ROOT, 'val')
EVAL_SETS = ['test_synth'] + [s for s in ['test_gold', 'test_adv'] if (ROOT / f'data/splits/{s}.jsonl').exists()]
evsets = {s: prompts.load_eval_set(s) for s in EVAL_SETS}
evsets = {s: (d.head(N_EVAL) if N_EVAL else d) for s, d in evsets.items()}
print(f'Huấn luyện: {len(train_ds):,} mẫu | kiểm định: {len(val_ds):,} | đánh giá:', {s: len(d) for s, d in evsets.items()})
if 'test_gold' not in EVAL_SETS:
    display(Markdown('> ⚠ Chưa có `test_gold` viết tay — kết quả trên `test_synth` là **trong phân phối**, chưa phải số chính thức.'))

## 2. Thiết lập
### 2.1 Siêu tham số
Mặc định r = 16, lr = 2e-4 (α = 2r, dropout 0,05, cosine, warmup 3%, batch hiệu dụng 16, loss chỉ trên lượt trợ lý).
Bật `RUN_OPTUNA` để dò r ∈ {8, 16, 32}, lr ∈ [5e-5, 3e-4] bằng TPE + MedianPruner trên ô rẻ nhất (Qwen QLoRA 4-bit, 30% dữ liệu).

In [ ]:
best_path = RES / 'optuna_best.json'
if RUN_OPTUNA and not best_path.exists():
    import optuna
    from transformers import TrainerCallback
    class Prune(TrainerCallback):
        def __init__(self, t): self.t = t
        def on_evaluate(self, args, state, control, metrics=None, **kw):
            self.t.report(metrics['eval_loss'], state.global_step)
            if self.t.should_prune(): raise optuna.TrialPruned()
    def objective(t):
        r, lr = t.suggest_categorical('r', [8, 16, 32]), t.suggest_float('lr', 5e-5, 3e-4, log=True)
        res = llm.train(ROOT, 'qwen', 'nf4', str(WORK / f'optuna/t{t.number}'), seed=SEED, lr=lr, r=r, max_steps=120,
                        data_frac=0.3, eval_steps=30, max_len=MAX_LEN, extra_callbacks=[Prune(t)])
        return res['eval_loss']
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED),
                                pruner=optuna.pruners.MedianPruner(n_startup_trials=2, n_warmup_steps=30))
    study.optimize(objective, n_trials=N_TRIALS)
    study.trials_dataframe().to_csv(RES / 'optuna_trials.csv', index=False)
    best_path.write_text(json.dumps({'best_params': study.best_params, 'best_val_loss': study.best_value}, indent=1))
HP = json.loads(best_path.read_text())['best_params'] if best_path.exists() else DEFAULT_HP
print('Siêu tham số dùng cho mọi ô:', HP)

### 2.2 Mô hình cơ sở (B0 zero-shot, mô hình gốc fp16)
Điểm mốc để đo phần cải thiện do fine-tuning.

In [ ]:
def eval_model(model, tok, system_name, modes=('closed',), contexts_fn=None, db_src=ROOT / 'data/kimochi.db'):
    """Sinh trên mọi tập đánh giá cho các chế độ: closed (không ngữ cảnh) · rag (thêm chunk) · agent (công cụ + chunk)."""
    rows = []
    for es, ev in evsets.items():
        for mode in modes:
            f = GEN / f'{system_name}_{mode}_{es}.jsonl'
            if f.exists():
                out = pd.read_json(f, lines=True)
            elif mode == 'agent':
                db = WORK / 'eval.db'; shutil.copy(db_src, db)
                ag = agent.run_batch(model, tok, list(ev['query']), db, contexts=[contexts_fn(q) for q in ev['query']])
                out = ev.assign(pred=[a['pred'] for a in ag], latency_batch_s=[a['latency_batch_s'] for a in ag],
                                tools_called=[a['tools_called'] for a in ag])
            else:
                msgs = [prompts.build_messages(q, 'rag' if mode == 'rag' else 'zero', docs=contexts_fn(q) if mode == 'rag' else None)
                        for q in ev['query']]
                preds, lat = llm.generate(model, tok, msgs, bs=8)
                out = ev.assign(pred=preds, latency_batch_s=lat)
            out = out.assign(system=f'{system_name}_{mode}', eval_set=es)
            out.to_json(f, orient='records', lines=True, force_ascii=False)
            s = metrics.summarize(metrics.score_generations(out))
            if mode == 'agent':
                s['tool_call_rate_price'] = float(out[out.intent == 'price_query'].tools_called.map(lambda t: 'get_menu_item' in t).mean())
            rows.append({'system': f'{system_name}_{mode}', 'eval_set': es, **s})
    return rows

baseline_rows = []
if RUN_BASELINE:
    for mk in MODELS:
        model, tok, _ = llm.load(mk, 'fp16')
        baseline_rows += [dict(r, system=r['system'].replace('_closed', ''), model=mk) for r in eval_model(model, tok, f'{mk}_B0')]
        del model, tok; llm.free()
    pd.DataFrame(baseline_rows).to_csv(RES / 'baseline.csv', index=False)
    display(pd.DataFrame(baseline_rows).round(3))

## 3. Fine-tuning: LoRA 16-bit → QLoRA 4-bit (→ QLoRA 8-bit tùy chọn)
- **LoRA 16-bit:** trọng số nền fp16 đóng băng, chỉ huấn luyện ma trận hạng thấp $B A$: $h = W_0x + \frac{\alpha}{r}BAx$.
- **QLoRA 4-bit:** trọng số nền lượng tử **NF4** + double quantization, tính toán fp16, bộ tối ưu `paged_adamw_8bit`.
- **QLoRA 8-bit (tùy chọn):** trọng số nền LLM.int8.

Mỗi ô ghi `result.json` (VRAM đỉnh, thời gian, val loss) + `history.json`; ô đã có thì bỏ qua.

In [ ]:
train_rows = []
for mk in MODELS:
    for method, q in METHODS.items():
        out = RUNS / f'{mk}_{q}_s{SEED}'
        if (out / 'result.json').exists():
            res = json.loads((out / 'result.json').read_text())
        else:
            print(f'▶ Huấn luyện {mk} · {method}')
            res = llm.train(ROOT, mk, q, str(out), seed=SEED, lr=HP['lr'], r=HP['r'], epochs=EPOCHS, max_len=MAX_LEN)
            if HF_USER:
                res['hub'] = llm.push_adapter(res['adapter'], f'{HF_USER}/kimochi-{mk}-{q}-s{SEED}')
                (out / 'result.json').write_text(json.dumps(res, indent=1))
        train_rows.append({**res, 'method': method, 'run': out.name})
        pd.DataFrame(train_rows).to_csv(RES / 'train_matrix.csv', index=False)
TM = pd.DataFrame(train_rows)
display(TM[['model', 'method', 'trainable_params', 'peak_vram_gb', 'train_minutes', 'steps', 'eval_loss', 'eval_ppl']].round(3))

In [ ]:
hist = {f"{r.model} · {r.method}": json.loads((RUNS / r.run / 'history.json').read_text()) for r in TM.itertuples()}
(RES / 'histories').mkdir(exist_ok=True)
for r in TM.itertuples():
    shutil.copy(RUNS / r.run / 'history.json', RES / 'histories' / f'{r.run}.json')
viz.loss_curves(hist, RES / 'loss_curves.png'); plt.show()
viz.train_resources(TM, RES / 'train_resources.png'); plt.show()

**Đọc biểu đồ.** Đường chấm đứng = bước có val loss nhỏ nhất (checkpoint được giữ nhờ `load_best_model_at_end`). Val loss tăng lại
trong khi train loss vẫn giảm → **quá khớp**, dừng sớm đã chặn. So sánh cặp LoRA ↔ QLoRA cùng mô hình: chênh VRAM là lợi ích
của lượng tử hóa, chênh val loss là cái giá phải trả (RQ2).

## 4. Đánh giá các mô hình fine-tune trên tập test (closed-book — đo riêng năng lực của mô hình)

In [ ]:
ft_rows = []
for r in TM.itertuples():
    model, tok, _ = llm.load(r.model, r.quant, adapter=str(RUNS / r.run / 'adapter'))
    ft_rows += [dict(x, model=r.model, method=r.method, run=r.run) for x in eval_model(model, tok, r.run)]
    del model, tok; llm.free()
FT = pd.DataFrame(ft_rows); FT.to_csv(RES / 'finetuned_eval.csv', index=False)
display(FT[['system', 'eval_set', 'rougeL', 'fact_em', 'hallucinated_price_rate', 'refusal_rate']].round(3))

## 5. Trực quan & phân tích kết quả

In [ ]:
ES = 'test_gold' if 'test_gold' in EVAL_SETS else 'test_synth'
def scored(system_mode, es=ES):
    return metrics.score_generations(pd.read_json(GEN / f'{system_mode}_{es}.jsonl', lines=True))
systems = ([f'{mk}_B0_closed' for mk in MODELS] if RUN_BASELINE else []) + [f'{r}_closed' for r in TM.run]
S = {s: scored(s) for s in systems}
tab, ci = {}, {}
for s, d in S.items():
    m, lo, hi = metrics.bootstrap_ci(d['rougeL']); ci[(s, 'ROUGE-L')] = (lo, hi)
    tab[s] = {'ROUGE-L': m, 'Từ chối': d['refused'].mean(), 'Bịa giá': d['hallucinated_price'].mean()}
T = pd.DataFrame(tab).T
T.to_csv(RES / f'ci_{ES}.csv'); display(T.round(3))
viz.metric_bars(T, ['ROUGE-L', 'Từ chối', 'Bịa giá'], ci, f'Mô hình cơ sở ↔ fine-tune ({ES}, khoảng tin cậy 95%)',
                RES / 'eval_models.png'); plt.show()

In [ ]:
q = TM.merge(T['ROUGE-L'].rename('rougeL'), left_on=TM.run + '_closed', right_index=True)
q['label'] = q.model + ' · ' + q.method
viz.quality_vs_vram(q, 'rougeL', RES / 'quality_vs_vram.png'); plt.show()
ih = viz.intent_heatmap(S, 'rougeL', RES / 'intent_heatmap.png'); plt.show()

In [ ]:
# So sánh ghép cặp (thay K-Fold): mô hình tốt nhất ↔ từng hệ thống khác
best_sys = T['ROUGE-L'].idxmax()
cmp = pd.DataFrame({s: metrics.paired_bootstrap(S[best_sys]['rougeL'].values, S[s]['rougeL'].values)
                    for s in S if s != best_sys}).T
cmp.to_csv(RES / 'paired_bootstrap.csv'); display(cmp.round(3))
noisy = {s: d.groupby('noisy_input')['rougeL'].mean() for s, d in S.items()}
display(pd.DataFrame(noisy).T.rename(columns={False: 'câu sạch', True: 'câu nhiễu'}).round(3))   # công bằng: nhóm câu nhiễu
lines = [f"- Mô hình tốt nhất (ROUGE-L, {ES}): **{best_sys}**."]
for mk in MODELS:
    a, b = f'{mk}_fp16_s{SEED}_closed', f'{mk}_nf4_s{SEED}_closed'
    if a in T.index and b in T.index:
        va = TM.set_index('run').loc[[a[:-7], b[:-7]], 'peak_vram_gb']
        lines.append(f"- {mk}: QLoRA 4-bit so với LoRA 16-bit: ROUGE-L {T.loc[b, 'ROUGE-L'] - T.loc[a, 'ROUGE-L']:+.3f}, "
                     f"VRAM {va.iloc[1] - va.iloc[0]:+.2f} GB (RQ2).")
for meth in METHODS.values():
    a, b = f'qwen_{meth}_s{SEED}_closed', f'llama_{meth}_s{SEED}_closed'
    if a in T.index and b in T.index:
        lines.append(f"- {meth}: Qwen − LLaMA = {T.loc[a, 'ROUGE-L'] - T.loc[b, 'ROUGE-L']:+.3f} ROUGE-L (RQ1).")
display(Markdown('**Phân tích tự động**\n' + '\n'.join(lines)))
worst = S[best_sys].sort_values('rougeL').head(10)[['query', 'reference', 'pred', 'intent']]
worst.to_csv(RES / 'error_cases.csv', index=False); display(worst)

**Gợi ý phân tích để đưa vào báo cáo (Phần 4–5):** (1) RQ1 – họ mô hình nào tốt hơn, chênh có nằm ngoài khoảng tin cậy không
(`paired_bootstrap.csv`, `p_not_better` < 0,05); (2) RQ2 – đánh đổi chất lượng ↔ VRAM (`quality_vs_vram.png`); (3) intent nào
yếu (`intent_heatmap.png`) và vì sao; (4) chênh lệch câu nhiễu ↔ câu sạch (công bằng); (5) phân nhóm 10 ca lỗi.
Lưu ý: closed-book **không có** công cụ / ngữ cảnh nên câu hỏi giá *phải* từ chối hoặc sai — §8 đo hệ thống đầy đủ.

## 6. Demo giao diện nhanh (mô hình tốt nhất, chưa có Hybrid-RAG đầy đủ)
Bật `RUN_QUICK_DEMO`; link `*.gradio.live` hiện bên dưới, notebook không bị chặn.

In [ ]:
BEST_RUN = best_sys.replace('_closed', '') if not best_sys.startswith(tuple(f'{m}_B0' for m in MODELS)) else TM.sort_values('eval_loss').run.iloc[0]
BEST = TM.set_index('run').loc[BEST_RUN]
print('Mô hình dùng cho các bước sau:', BEST_RUN)
if RUN_QUICK_DEMO:
    os.environ.update({'KIMOCHI_BACKEND': 'hf', 'KIMOCHI_MODEL': BEST.model, 'KIMOCHI_QUANT': BEST.quant,
                       'KIMOCHI_ADAPTER': str(RUNS / BEST_RUN / 'adapter')})
    sys.path.insert(0, str(ROOT / 'demo')); import app
    app.demo.queue().launch(share=True, prevent_thread_lock=True)

In [ ]:
if RUN_QUICK_DEMO:
    app.demo.close(); app._BACKEND = None; llm.free()   # đóng demo, giải phóng GPU trước §7

## 7. Xây dựng hệ thống Hybrid-RAG
Đơn vị truy xuất: chunk ≤ 600 ký tự (giống dữ liệu RAFT). Thành phần: **BM25** trên văn bản bỏ dấu + bigram · **TF-IDF n-gram ký tự** ·
**dense `BAAI/bge-m3`** · hợp nhất **RRF** (k = 60) · **reranker `BAAI/bge-reranker-v2-m3`** (top-20) · **lọc theo tên món** (metadata).
Đánh giá: câu hỏi test có gắn món → tài liệu đúng = tài liệu của món đó.

In [ ]:
dense, rer = retriever.load_models()
RAG = retriever.HybridRetriever(retriever.load_docs(), dense, rer)
rows = []
ev = retriever.eval_set('test_synth')
for cfg in retriever.CONFIGS:
    if ('dense' in cfg and dense is None) or ('rerank' in cfg and rer is None):
        rows.append({'config': cfg, 'status': 'skipped'}); continue
    t0 = time.perf_counter(); ranked = [RAG.search(q, cfg, k=10) for q in ev['query']]; dt = (time.perf_counter() - t0) / len(ev)
    m = retriever.metrics(ranked, list(ev['gold']))
    mn = retriever.metrics([x for x, n in zip(ranked, ev['noisy']) if n], list(ev.loc[ev['noisy'], 'gold']))
    rows.append({'config': cfg, 'status': 'ok', 'n': len(ev), **m, 'Recall@5_noisy': mn['Recall@5'], 'ms_per_query': dt * 1000})
RR = pd.DataFrame(rows); RR.to_csv(RES / 'retrieval_test_synth.csv', index=False)
display(RR.round(3)); viz.retrieval_bars(RR, RES / 'retrieval_ablation.png'); plt.show()
ok = RR[RR.status == 'ok']
RAG_CFG = ok.sort_values(['MRR@10', 'ms_per_query'], ascending=[False, True]).config.iloc[0]
print('Cấu hình truy xuất được chọn (MRR@10 cao nhất):', RAG_CFG)
ctx = lambda q: RAG.search_chunks(q, RAG_CFG, k=3)

In [ ]:
ex = retriever.explain_bm25(RAG, 'com cuon kieu cali bao nhieu tien')       # XAI cho bộ truy xuất
print(ex['chunk_id'], ex['title'], round(ex['score'], 2)); display(pd.DataFrame(ex['terms'], columns=['token', 'đóng góp BM25']))

## 8. Đánh giá hệ thống đầy đủ & so sánh
Cùng mô hình tốt nhất ở ba chế độ: **closed-book** → **+ Hybrid-RAG** → **+ RAG + Agent** (công cụ SQL chạy trên bản sao DB),
so với **mô hình gốc + RAG** (không fine-tune, trả lời RQ3). Thêm định tuyến công cụ, kịch bản đặt bàn nhiều lượt và LLM-as-judge.

In [ ]:
sys_rows = []
model, tok, _ = llm.load(BEST.model, BEST.quant, adapter=str(RUNS / BEST_RUN / 'adapter'))
sys_rows += eval_model(model, tok, BEST_RUN, modes=('closed', 'rag', 'agent'), contexts_fn=ctx)
lat1 = llm.latency_single(model, tok, [prompts.build_messages(q, 'rag', docs=ctx(q)) for q in evsets['test_synth'].head(30)['query']])
sys_rows.append({'system': f'{BEST_RUN}_rag', 'eval_set': 'latency30', 'latency_p50': float(np.median(lat1)),
                 'latency_p95': float(np.quantile(lat1, .95))})
gen_fn = lambda msgs: llm.generate(model, tok, [msgs], bs=1)[0][0]
sc_path = ROOT / 'data/splits/scenarios.jsonl'
scs = [json.loads(l) for l in sc_path.read_text(encoding='utf-8').splitlines() if l.strip()] if sc_path.exists() else agent_eval.scenarios_from_v2_val(ROOT)
kind = 'viết tay' if sc_path.exists() else 'PROXY (val v2)'
(WORK / 'sc').mkdir(exist_ok=True)
SC = pd.DataFrame([agent_eval.run_scenario(s, gen_fn, ROOT / 'data/kimochi.db', WORK / 'sc') for s in scs])
SC.to_json(RES / 'agent_scenarios.jsonl', orient='records', lines=True, force_ascii=False)
(RES / 'agent_summary.json').write_text(json.dumps({'kind': kind, 'n': len(SC), 'task_success': SC.success.mean(),
    'db_state_ok': SC.db_state_ok.mean(), 'hitl_ok': SC.hitl_ok.mean(), 'avg_tool_calls': SC.n_tool_calls.mean()}, indent=1, default=float))
del model, tok; llm.free()
base_mk = BEST.model
model, tok, _ = llm.load(base_mk, 'fp16')                                         # mô hình gốc + RAG (không fine-tune)
sys_rows += eval_model(model, tok, f'{base_mk}_B2', modes=('rag',), contexts_fn=ctx)
del model, tok; llm.free()
SYS = pd.DataFrame(sys_rows)
FT_all = pd.concat([pd.read_csv(RES / 'finetuned_eval.csv'), SYS[SYS.system.str.startswith(BEST_RUN)]]).drop_duplicates(['system', 'eval_set'], keep='last')
FT_all.to_csv(RES / 'finetuned_eval.csv', index=False)
pd.concat([pd.read_csv(RES / 'baseline.csv') if (RES / 'baseline.csv').exists() else pd.DataFrame(),
           SYS[SYS.system.str.startswith(f'{base_mk}_B2')].assign(system=f'{base_mk}_B2', model=base_mk)]
          ).drop_duplicates(['system', 'eval_set'], keep='last').to_csv(RES / 'baseline.csv', index=False)
display(SYS.round(3)); print('Kịch bản:', kind, json.loads((RES / 'agent_summary.json').read_text()))

In [ ]:
order = [f'{base_mk}_B0_closed', f'{base_mk}_B2_rag', f'{BEST_RUN}_closed', f'{BEST_RUN}_rag', f'{BEST_RUN}_agent']
names = {order[0]: 'Gốc', order[1]: 'Gốc + RAG', order[2]: 'Fine-tune', order[3]: 'Fine-tune + RAG', order[4]: 'Fine-tune + RAG + Agent'}
E2 = {names[s]: scored(s, 'test_synth') for s in order if (GEN / f'{s}_test_synth.jsonl').exists()}
cmpT, ci2 = {}, {}
for n, d in E2.items():
    for col, key in [('rougeL', 'ROUGE-L'), ('fact_ok', 'Fact-EM giá'), ('refused', 'Từ chối')]:
        m, lo, hi = metrics.bootstrap_ci(d[col]); ci2[(n, key)] = (lo, hi); cmpT.setdefault(n, {})[key] = m
    cmpT[n]['Bịa giá'] = d['hallucinated_price'].mean()
C2 = pd.DataFrame(cmpT).T; C2.to_csv(RES / 'system_comparison.csv'); display(C2.round(3))
viz.metric_bars(C2, ['ROUGE-L', 'Fact-EM giá', 'Bịa giá'], ci2, 'So sánh hệ thống trên test_synth (CI 95%)', RES / 'system_comparison.png'); plt.show()
ag = pd.read_json(GEN / f'{BEST_RUN}_agent_test_synth.jsonl', lines=True)
rep = agent_eval.routing_report(ag)
rep['confusion'].to_csv(RES / 'tool_routing_confusion.csv'); rep['per_label'].to_csv(RES / 'tool_routing_per_label.csv')
(RES / 'tool_routing_summary.json').write_text(json.dumps({k: rep[k] for k in ('accuracy', 'macro_f1', 'business_cost', 'cost_per_100')}, indent=1))
agent_eval.plot_confusion(rep['confusion'], RES / 'tool_routing_confusion.png'); display(Image(RES / 'tool_routing_confusion.png', width=520))
print({k: round(rep[k], 3) for k in ('accuracy', 'macro_f1', 'cost_per_100')})

In [ ]:
JUDGE_N = 100   # LLM-as-judge (Gemini) — cần secret GEMINI_API_KEY; hiệu chỉnh với data/splits/human_scores.csv nếu có
if os.environ.get('GEMINI_API_KEY'):
    from scipy.stats import spearmanr
    jr = []
    for n, s in [('Gốc + RAG', order[1]), ('Fine-tune + RAG', order[3]), ('Fine-tune + RAG + Agent', order[4])]:
        d = scored(s, ES).head(JUDGE_N)
        for r in d.itertuples():
            sc_, why = metrics.judge(r.query, r.reference, r.pred); jr.append({'id': r.id, 'system': s, 'judge': sc_, 'reason': why}); time.sleep(.4)
    J = pd.DataFrame(jr); J.to_csv(RES / 'judge_scores.csv', index=False); display(J.groupby('system').judge.agg(['mean', 'count']).round(2))
    hs = ROOT / 'data/splits/human_scores.csv'
    if hs.exists():
        h = pd.read_csv(hs).merge(J, on=['id', 'system']); print('Spearman judge ↔ người:', spearmanr(h.human_score, h.judge), 'n =', len(h))
else:
    print('Bỏ qua judge (chưa có GEMINI_API_KEY)')

**Gợi ý phân tích:** RAG có cải thiện cả mô hình gốc lẫn mô hình fine-tune không (RQ3)? Agent có đưa **Fact-EM giá** lên gần 100% và
**bịa giá** về 0 không (giá đọc từ DB)? Nhầm lẫn định tuyến nào tốn chi phí nhất (ma trận nhầm lẫn)? Tỷ lệ hoàn thành kịch bản
đặt bàn và tuân thủ xác nhận trước khi ghi (`agent_summary.json`)? Độ trễ p95 so với KPI ≤ 5 s?

## 9. Xuất mô hình hoàn chỉnh cho MVP
Gộp adapter vào trọng số fp16 (`merge_and_unload`) → thư mục safetensors độc lập (không cần PEFT khi chạy), kiểm tra nạp lại
ở NF4 và trả lời đúng, tùy chọn đẩy lên Hugging Face Hub, đóng gói **MVP bundle** (mô hình + app + kho tri thức + DB + hướng dẫn).

In [ ]:
EXPORT = WORK / 'kimochi-mvp-model'
if not (EXPORT / 'kimochi_export.json').exists():
    info = llm.export_merged(BEST.model, str(RUNS / BEST_RUN / 'adapter'), str(EXPORT))
else:
    info = json.loads((EXPORT / 'kimochi_export.json').read_text())
info.update({'run': BEST_RUN, 'method': BEST.method, 'retrieval_config': RAG_CFG}); (EXPORT / 'kimochi_export.json').write_text(json.dumps(info, indent=1))
print(info)
m2, t2, _ = llm.load(str(EXPORT), 'nf4')                                         # kiểm tra nạp lại + chạy agent 1 câu
db = WORK / 'smoke.db'; shutil.copy(ROOT / 'data/kimochi.db', db)
r = agent.run([agent.system_with_tools(), {'role': 'user', 'content': 'com luon bao nhieu tien vay shop'}],
              lambda m: llm.generate(m2, t2, [m], bs=1)[0][0], db)
print('Trả lời:', r['answer']); print('Công cụ:', [t['call'] for t in r['trace']])
del m2, t2; llm.free()
if HF_USER:
    print(llm.push_adapter(str(EXPORT), f'{HF_USER}/kimochi-mvp'))

In [ ]:
BUNDLE = WORK / 'kimochi-mvp'
if BUNDLE.exists(): shutil.rmtree(BUNDLE)
for sub in ['demo', 'code/src', 'data/kb']:
    shutil.copytree(ROOT / sub, BUNDLE / sub, ignore=shutil.ignore_patterns('__pycache__'))
shutil.copy(ROOT / 'data/kimochi.db', BUNDLE / 'data/kimochi.db')
model_ref = f'{HF_USER}/kimochi-mvp' if HF_USER else '../kimochi-mvp-model'
(BUNDLE / 'run_mvp.sh').write_text(f"""#!/usr/bin/env bash
# Chạy MVP: cần GPU ≥ 8 GB (nạp NF4). Đổi KIMOCHI_MODEL nếu đặt mô hình ở nơi khác.
pip install -r demo/requirements.txt
KIMOCHI_BACKEND=hf KIMOCHI_MODEL={model_ref} KIMOCHI_QUANT=nf4 KIMOCHI_ADAPTER= python demo/app.py
""")
(BUNDLE / 'MVP_README.md').write_text(f"""# Kimochi Sushi — MVP
Mô hình: {info['base']} + adapter {BEST.method} (gộp, fp16, {info['size_gb']} GB) · truy xuất: `{RAG_CFG}`.
Chạy: `bash run_mvp.sh` → http://localhost:7860. Bàn / đặt bàn là dữ liệu mô phỏng; dị ứng chỉ tham khảo.
""")
shutil.make_archive(str(WORK / 'kimochi-mvp-bundle'), 'zip', BUNDLE)
print('Gói MVP:', WORK / 'kimochi-mvp-bundle.zip', '| mô hình:', EXPORT)

## 10. Demo MVP
Chạy app với **mô hình đã xuất** (không cần adapter). Link `*.gradio.live` dùng để quay video 2–3 phút (kịch bản: `demo/README.md`) hoặc cho người dùng thật thử. Dừng ô này để tắt.

In [ ]:
if RUN_MVP_DEMO:
    os.environ.update({'KIMOCHI_BACKEND': 'hf', 'KIMOCHI_MODEL': str(EXPORT), 'KIMOCHI_QUANT': 'nf4', 'KIMOCHI_ADAPTER': ''})
    sys.path.insert(0, str(ROOT / 'demo'))
    import importlib, app; importlib.reload(app)
    app.demo.queue().launch(share=True, debug=True)